<a href="https://colab.research.google.com/github/MANI-WEBDEVE/RAG_SYSTEM/blob/main/RAG_CHUNK_CONCEPT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os

if "COLAB_GPU" in os.environ:
  print('ok')
  # !pip install -U torch # requires torch 2.1.1+ (for efficient sdpa implementation)
  !pip install PyMuPDF
  !pip install sentence-transformers # for embedding models
  !pip install tqdm
  !pip install accelerate
  !pip install bitsandbytes
  !pip install flash-attn --no-build-isolation

ok


In [ ]:
!pip uninstall -y torch torchvision torchaudio transformers sentence-transformers
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -U transformers sentence-transformers

Found existing installation: torch 2.9.0+cu126
Uninstalling torch-2.9.0+cu126:
  Successfully uninstalled torch-2.9.0+cu126
Found existing installation: torchvision 0.24.0+cu126
Uninstalling torchvision-0.24.0+cu126:
  Successfully uninstalled torchvision-0.24.0+cu126
Found existing installation: torchaudio 2.9.0+cu126
Uninstalling torchaudio-2.9.0+cu126:
  Successfully uninstalled torchaudio-2.9.0+cu126
Found existing installation: transformers 4.57.6
Uninstalling transformers-4.57.6:
  Successfully uninstalled transformers-4.57.6
Found existing installation: sentence-transformers 5.2.2
Uninstalling sentence-transformers-5.2.2:
  Successfully uninstalled sentence-transformers-5.2.2
Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.4/780.4 MB 1.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 82.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 92.9 MB/s eta 0:00:00
     ━━━━━

## Download the Docs

In [ ]:
import os
import requests

pdf_path = "ML.pdf"

# Download pdf if doesn`t exits
if not os.path.exists(pdf_path):
  print("File does not exist")

  # The URL of the PDF of Download
  url = 'https://shashwatwork.github.io/assets/files/ml_ebook.pdf'

  filename=pdf_path

  # send the GET request to download the PDF

  response = requests.get(url)

  # check if the request was successful
  if response.status_code == 200:
    # open a file in binary write mode and save the contant it
    with open(filename, 'wb') as file:
      file.write(response.content)
    print(f'write the file has been downloaded and saved as {filename}')
  else:
    print(f'the file failed to download status code: {response.status_code}')
else:
  print(f"file {pdf_path} exists")


File does not exist
write the file has been downloaded and saved as ML.pdf


## Extract the text from documents

In [ ]:
from tqdm.auto import tqdm
import fitz

def text_formater(text:str) -> str:
  """ Perform the minner formatting """
  cleaned_text=text.replace("\n", " ").strip()

  return cleaned_text
# Note: focus on only text rather than images and table

def open_and_read_pdf(pdf_path:str)->list[dict]:
  """
  Open : the pdf and read the text page by page

  Parameter get function:
    pdf_path the file path your pdf to open and read the document
  Return:
    list of and dictionary fromat [{}, {}] to return this function

  """
  docs = fitz.open(pdf_path)
  page_and_text=[]

  for page_number , page in tqdm(enumerate(docs)):
    text=page.get_text()
    text=text_formater(text)
    page_and_text.append({
        "page_number": page_number,
        "page_total_words": len(text.split(' ')),
        "page_total_char": len(text),
        "page_total_sentence": len(text.split('. ')),
        "page_total_token": len(text) /4,
        "text": text
    })

  return page_and_text


page_and_text = open_and_read_pdf(pdf_path=pdf_path)
page_and_text[:20]




0it [00:00, ?it/s]

[{'page_number': 0,
  'page_total_words': 1,
  'page_total_char': 0,
  'page_total_sentence': 1,
  'page_total_token': 0.0,
  'text': ''},
 {'page_number': 1,
  'page_total_words': 1,
  'page_total_char': 0,
  'page_total_sentence': 1,
  'page_total_token': 0.0,
  'text': ''},
 {'page_number': 2,
  'page_total_words': 30,
  'page_total_char': 237,
  'page_total_sentence': 1,
  'page_total_token': 59.25,
  'text': 'Aurélien Géron Hands-on Machine Learning with Scikit-Learn, Keras, and TensorFlow Concepts, Tools, and Techniques to Build Intelligent Systems SECOND EDITION Boston Farnham Sebastopol Tokyo Beijing Boston Farnham Sebastopol Tokyo Beijing'},
 {'page_number': 3,
  'page_total_words': 249,
  'page_total_char': 1777,
  'page_total_sentence': 14,
  'page_total_token': 444.25,
  'text': '978-1-492-03264-9 [LSI] Hands-on Machine Learning with Scikit-Learn, Keras, and TensorFlow by Aurélien Géron Copyright © 2019 O’Reilly Media. All rights reserved. Printed in the United States of Am

In [ ]:
import random
random_Sam= random.sample(page_and_text, k=5)

In [ ]:
random_Sam

[{'page_number': 113,
  'page_total_words': 218,
  'page_total_char': 1363,
  'page_total_sentence': 10,
  'page_total_token': 340.75,
  'text': '5 You can use the shift() function from the scipy.ndimage.interpolation module. For example, shift(image, [2, 1], cval=0) shifts the image 2 pixels down and 1 pixel to the right. Let’s take a peek at an image from the test set (yes, we’re snooping on the test data, so you should be frowning right now): On the left is the noisy input image, and on the right is the clean target image. Now let’s train the classifier and make it clean this image: knn_clf.fit(X_train_mod, y_train_mod) clean_digit = knn_clf.predict([X_test_mod[some_index]]) plot_digit(clean_digit) Looks close enough to the target! This concludes our tour of classification. Hopefully you should now know how to select good metrics for classification tasks, pick the appropriate precision/recall tradeoff, compare classifiers, and more generally build good classification systems for a v

In [ ]:
import pandas as pd

df=pd.DataFrame(page_and_text)

In [ ]:
df

,page_number,page_total_words,page_total_char,page_total_sentence,page_total_token,text
0,0,1,0,1,0.00,
1,1,1,0,1,0.00,
2,2,30,237,1,59.25,Aurélien Géron Hands-on Machine Learning with ...
3,3,249,1777,14,444.25,978-1-492-03264-9 [LSI] Hands-on Machine Learn...
4,4,2467,3246,90,811.50,Table of Contents 1. The Machine Learning Land...
...,...,...,...,...,...,...
274,274,190,1131,10,282.75,Figure 9-22. Bayesian Gaussian mixture model P...
275,275,350,1771,13,442.75,Bayes’ theorem (Equation 9-2) tells us how to ...
276,276,363,2263,15,565.75,"In practice, there are different techniques to..."
277,277,440,2785,19,696.25,Other Anomaly Detection and Novelty Detection ...


In [ ]:
df.describe()

,page_number,page_total_words,page_total_char,page_total_sentence,page_total_token
count,279.000000,279.000000,279.000000,279.000000,279.000000
mean,139.000000,335.738351,1750.713262,13.591398,437.678315
std,80.684571,408.443051,691.596170,12.272303,172.899043
min,0.000000,1.000000,0.000000,1.000000,0.000000
25%,69.500000,222.000000,1367.500000,9.000000,341.875000
50%,139.000000,302.000000,1786.000000,12.000000,446.500000
75%,208.500000,356.500000,2124.500000,15.000000,531.125000
max,278.000000,3948.000000,4918.000000,124.000000,1229.500000


## Step no 3 Chunking Section
---
How to implement five methods of chunking. Fixed , Semantic , structural , recursive, LLM based.

## First Method is Fixed size of Chunk our data

In [ ]:
def chunk_size(text:str, chunk_size:int = 500) -> list:
  """
  Split the text to chunk size.
  """

  chunks=[]
  words=text.split()
  current_chunk=""

  for word in words:
    # if check the word before adding excced chunk
    if len(current_chunk) + len(word) + 2 <= chunk_size:
      current_chunk += (word + " ")
    else:
      chunks.append(current_chunk.strip())
      current_chunk += word + " "

  if current_chunk:
      chunks.append(current_chunk.strip())

  return chunks


In [ ]:
chunk_ex = []
current_chunk_ex = ""

ex_word = "this is my car"

In [ ]:
for word in ex_word.split():
  if len(current_chunk_ex) + len(word) + 1 <= 3:
    print(word)
    current_chunk_ex += (word + " ")
    print(word + " ")
  else :
    print('lo')

lo
lo
lo
lo


In [ ]:
current_chunk_ex

't h i s   i s   m y   c a r this is my car this is my car '

In [ ]:
chunk_ex.append(ex_word.split())

In [ ]:
chunk_ex

[['this', 'is', 'my', 'car']]